# HPA irrigation moisture-flow table

This notebook decodes annual RECON source-to-sink moisture flows and writes the irrigation-weighted contribution from every HPA source polygon with irrigation fraction greater than 0.1 to every CONUS sink polygon. The moisture-flow CSV columns are `cellid_source`, `cellid_sink`, and `moistureflow_volume` (m$^3$). It also writes annual precipitation volumes for each CONUS sink cell with columns `cellid` and `precipitation_m3`.

## Imports

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import shapefile
import xarray as xr
from matplotlib.path import Path as MatplotlibPath
from pyproj import CRS, Transformer

## Variables

In [2]:
home = Path.home()
current_dir = Path.cwd()
repository_root = current_dir if (current_dir / "data").exists() else current_dir.parent

input_file = home / "OneDrive - University of Kansas" / "Research" / "MoistureRecycling" / "RECON" / "RECON_moisture_flows_0.5.nc"
precipitation_volume_file = home / "OneDrive - University of Kansas" / "Research" / "MoistureRecycling" / "RECON" / "RECON_ERA5_avgYear_0.5_volumes.nc"
source_polygon_file = repository_root / "data" / "ERA5-HPA_LANID_AverageIrrigation.shp"
sink_polygon_file = repository_root / "data" / "ERA5_grid" / "ERA5_grid_CONUS.shp"
moistureflow_table_path = repository_root / "data" / "RECON_01_HPAirrigationMoistureFlow.csv"
precipitation_table_path = repository_root / "data" / "RECON_01_CONUStotalPrecipitationVolume.csv"

irrigation_threshold = 0.1  # percent irrigated to count as irrigated pixel
batch_size = 32
ymax = 122079329.40990189  # Maximum moisture flow value
ymin = 10**-3  # Minimum moisture flow value

## Load RECON data

In [3]:
if not input_file.exists():
    raise FileNotFoundError(f"RECON moisture-flow file not found: {input_file}")
if not precipitation_volume_file.exists():
    raise FileNotFoundError(f"RECON precipitation-volume file not found: {precipitation_volume_file}")

dataset = xr.open_dataset(input_file)
moisture_flow = dataset["moisture_flow"]
precipitation_volumes = xr.open_dataset(precipitation_volume_file)
annual_precipitation = precipitation_volumes["ERA5_TP_averageyear"]
if not np.array_equal(dataset.targetlat.values, annual_precipitation.lat.values) or not np.array_equal(
    dataset.targetlon.values, annual_precipitation.lon.values
):
    raise ValueError("RECON target grid does not match the precipitation-volume grid.")

## Match shapefile polygons to ERA5 cells

Each aligned polygon is matched to the ERA5 grid center at its bounding-box center. The containment check ensures that the matched grid center is actually within the polygon, including polygons with holes.

In [4]:
def cellid_value(value):
    numeric_value = float(value)
    return int(numeric_value) if numeric_value.is_integer() else numeric_value


def polygon_rings(shape, transformer):
    if not shape.points:
        return []

    part_starts = list(shape.parts) + [len(shape.points)]
    rings = []
    for start, end in zip(part_starts[:-1], part_starts[1:]):
        x, y = np.asarray(shape.points[start:end]).T
        longitude, latitude = transformer.transform(x, y)
        rings.append(np.column_stack((longitude, latitude)))
    return rings


def match_polygons_to_grid(polygon_file, grid_lats, grid_lons, irrigation_fields=()):
    polygon_file = Path(polygon_file)
    if polygon_file.suffix.lower() != ".shp" or not polygon_file.exists():
        raise FileNotFoundError(f"Polygon shapefile not found: {polygon_file}")

    prj_file = polygon_file.with_suffix(".prj")
    if not prj_file.exists():
        raise FileNotFoundError(f"Polygon CRS file not found: {prj_file}")

    reader = shapefile.Reader(str(polygon_file))
    field_names = [field[0] for field in reader.fields[1:]]
    field_lookup = {name.lower(): name for name in field_names}
    if "cellid" not in field_lookup:
        raise ValueError(f"{polygon_file.name} must contain a cellid field.")

    irrigation_field = next(
        (field_lookup[name.lower()] for name in irrigation_fields if name.lower() in field_lookup),
        None,
    )
    if irrigation_fields and irrigation_field is None:
        raise ValueError(
            f"{polygon_file.name} must contain one of {', '.join(irrigation_fields)}."
        )

    transformer = Transformer.from_crs(
        CRS.from_wkt(prj_file.read_text()), CRS.from_epsg(4326), always_xy=True
    )
    grid_lons_wgs84 = (np.asarray(grid_lons) + 180) % 360 - 180
    rows = []
    matched_cells = set()

    for shape_record in reader.iterShapeRecords():
        attributes = dict(zip(field_names, shape_record.record))
        irrigation_fraction = (
            float(attributes[irrigation_field]) if irrigation_field is not None else None
        )
        if irrigation_fraction is not None and irrigation_fraction <= irrigation_threshold:
            continue

        rings = polygon_rings(shape_record.shape, transformer)
        if not rings:
            raise ValueError(f"Empty geometry for cellid {attributes[field_lookup['cellid']]}")

        xmin, ymin, xmax, ymax = shape_record.shape.bbox
        longitude, latitude = transformer.transform((xmin + xmax) / 2, (ymin + ymax) / 2)
        lat_index = int(np.abs(grid_lats - latitude).argmin())
        lon_index = int(np.abs((grid_lons_wgs84 - longitude + 180) % 360 - 180).argmin())
        grid_point = (grid_lons_wgs84[lon_index], grid_lats[lat_index])

        contained = False
        for ring in rings:
            contained ^= MatplotlibPath(ring).contains_point(grid_point)
        if not contained:
            raise ValueError(
                f"ERA5 grid center {grid_point} is not inside polygon cellid "
                f"{attributes[field_lookup['cellid']]}"
            )

        grid_cell = (lat_index, lon_index)
        if grid_cell in matched_cells:
            raise ValueError(f"Multiple polygons match ERA5 cell {grid_cell} in {polygon_file.name}")
        matched_cells.add(grid_cell)
        rows.append(
            {
                "cellid": cellid_value(attributes[field_lookup["cellid"]]),
                "lat_index": lat_index,
                "lon_index": lon_index,
                "irrigation_fraction": irrigation_fraction,
            }
        )

    if not rows:
        raise ValueError(f"No polygons matched in {polygon_file.name}")
    return pd.DataFrame(rows)

## Calculate and write source-to-sink volumes

In [5]:
sources = match_polygons_to_grid(
    source_polygon_file,
    dataset.sourcelat.values,
    dataset.sourcelon.values,
    irrigation_fields=("prc_irrigated", "irr_avgPrc"),
)
sinks = match_polygons_to_grid(
    sink_polygon_file,
    dataset.targetlat.values,
    dataset.targetlon.values,
)
print(f"Selected {len(sources):,} irrigated source cells and {len(sinks):,} CONUS sink cells.")

sink_cell_indices = xr.DataArray(np.arange(len(sinks)), dims="sink_cell")
sink_lat_indices = xr.DataArray(sinks["lat_index"].to_numpy(), dims="sink_cell")
sink_lon_indices = xr.DataArray(sinks["lon_index"].to_numpy(), dims="sink_cell")
sink_ids = sinks["cellid"].to_numpy()

precipitation_table_path.parent.mkdir(parents=True, exist_ok=True)
sink_precipitation_table = pd.DataFrame(
    {
        "cellid": sink_ids,
        "precipitation_m3": annual_precipitation.isel(
            lat=sink_lat_indices, lon=sink_lon_indices
        ).values,
    }
)
sink_precipitation_table.to_csv(precipitation_table_path, index=False)
if len(sink_precipitation_table) != len(sinks):
    raise RuntimeError("Not all CONUS sink cells were written to the precipitation table.")
print(f"Wrote {len(sink_precipitation_table):,} rows to {precipitation_table_path}")

if moistureflow_table_path.exists():
    moistureflow_table_path.unlink()

for batch_start in range(0, len(sources), batch_size):
    batch = sources.iloc[batch_start : batch_start + batch_size]
    encoded_flows = moisture_flow.isel(
        sourcelat=xr.DataArray(batch["lat_index"].to_numpy(), dims="source_cell"),
        sourcelon=xr.DataArray(batch["lon_index"].to_numpy(), dims="source_cell"),
        targetlat=sink_lat_indices,
        targetlon=sink_lon_indices,
    )
    decoded_flows = xr.where(
        encoded_flows == 0,
        0,
        10 ** (
            ((encoded_flows - 1) / 254) * (np.log10(ymax) - np.log10(ymin))
            + np.log10(ymin)
        ),
    ).fillna(0).transpose("source_cell", "sink_cell")
    irrigation_weighted_flows = decoded_flows.values * batch["irrigation_fraction"].to_numpy()[:, None]

    flow_table_batch = pd.DataFrame(
        {
            "cellid_source": np.repeat(batch["cellid"].to_numpy(), len(sinks)),
            "cellid_sink": np.tile(sink_ids, len(batch)),
            "moistureflow_volume": irrigation_weighted_flows.ravel(),
        }
    )
    flow_table_batch.to_csv(
        moistureflow_table_path,
        mode="a",
        header=batch_start == 0,
        index=False,
    )
    print(f"Wrote sources {batch_start + 1:,}-{batch_start + len(batch):,} of {len(sources):,}")

expected_rows = len(sources) * len(sinks)
written_rows = sum(1 for _ in moistureflow_table_path.open()) - 1
if written_rows != expected_rows:
    raise RuntimeError(f"Expected {expected_rows:,} rows, but wrote {written_rows:,}.")

print(f"Wrote {written_rows:,} rows to {moistureflow_table_path}")
pd.read_csv(moistureflow_table_path, nrows=10)

Selected 92 irrigated source cells and 3,180 CONUS sink cells.
Wrote 3,180 rows to C:\Users\s947z036\WorkGits\MoistureRecyclingRECON\data\RECON_01_CONUSSinkPrecipitation.csv
Wrote sources 1-32 of 92
Wrote sources 33-64 of 92
Wrote sources 65-92 of 92
Wrote 292,560 rows to C:\Users\s947z036\WorkGits\MoistureRecyclingRECON\data\RECON_01_HPAirrigationMoistureFlow.csv


,cellid_source,cellid_sink,moistureflow_volume
0,12116,9394,205695.143067
1,12116,9550,1.966443
2,12116,9551,1.075950
3,12116,9552,0.002861
4,12116,9553,0.003164
5,12116,9554,0.002861
6,12116,9555,0.435470
7,12116,9556,133.938561
8,12116,9557,4.858644
9,12116,9558,40.098420
